# Part 2: How Real VLAs Represent Actions

## Notebook 3 — ACT (Action Chunking Transformer)

ACT (Zhao et al., RSS 2023) predicts **continuous action chunks** using a Conditional Variational Autoencoder (CVAE). No tokenization — actions are raw continuous vectors.

We load ACT from leRobot v0.6.0 and inspect its action handling.


### 1. Load ACT configuration

ACT is a policy that predicts chunks of actions directly. No tokenization, no discretization — just continuous regression.


In [ ]:
from lerobot.policies.act.configuration_act import ACTConfig

cfg = ACTConfig()
print(f"Policy type: ACT (Action Chunking Transformer)")
print(f"Chunk size:        {cfg.chunk_size}")  # 100
print(f"Action steps:      {cfg.n_action_steps}")  # 100
print(f"Input shapes:      {cfg.input_shapes}")
print(f"Output shapes:     {cfg.output_shapes}")


### 2. Action representation: pure continuous

ACT outputs a tensor of shape `(batch, chunk_size, action_dim)`. Each value is a raw float — no binning, no discretization, no tokens.


In [ ]:
import torch

# Simulate what ACT outputs
batch_size = 1
chunk_size = cfg.chunk_size  # 100
action_dim = 7  # typical: x, y, z, roll, pitch, yaw, gripper

actions = torch.randn(batch_size, chunk_size, action_dim)
print(f"ACT action shape:  {actions.shape}")
print(f"Total values:      {actions.numel()}")  # 700
print(f"Value range:       [{actions.min():.2f}, {actions.max():.2f}]")
print(f"Data type:         {actions.dtype}")

# Compare: if this were RT-1 style binning (256 bins/dim)
tokens_if_binned = chunk_size * action_dim  # 700 tokens
print(f"\nIf binning (256 bins/dim): {tokens_if_binned} tokens per chunk")print(f"ACT uses 0 tokens — continuous vectors instead")


### 3. CVAE: the stochastic action head

ACT doesn't just regress — it samples from a learned distribution. The CVAE encodes observations into a latent distribution (μ, σ), samples z, and decodes into action chunks. This captures multi-modal action distributions (e.g., you could go left OR right around an obstacle).


In [ ]:
# ACT uses a CVAE (Conditional Variational Autoencoder)
# Encoder: observation -> latent distribution (mu, sigma)
# Sample: z ~ N(mu, sigma)
# Decoder: z -> action chunk (chunk_size, action_dim)

# The loss = reconstruction_loss + kl_divergence
# This allows ACT to model MULTIPLE valid action trajectories
# for the same observation — multimodal action distributions.

print("ACT CVAE Flow:")
print("  Observation -> Encoder -> (μ, σ) -> Sample z")
print("  z -> Decoder -> Action chunk (100 × 7 continuous values)")

# Compare with tokenization-based approaches:
print("\nContrast with tokenization VLAs:")
print("  RT-2: Observation -> LLM -> Token IDs -> Binned action values")
print("  pi0-FAST: Observation -> VLM -> FAST tokens -> Inverse DCT")


### 4. Temporal ensemble (smoothing)

ACT uses temporal ensembling to smooth consecutive action chunks. Overlapping chunks are averaged with exponential weighting.


In [ ]:
# Temporal ensemble: when chunks overlap, average them
# If chunk_1 predicts actions [a0..a99] and chunk_2 predicts [a50..a149],
# actions a50..a99 are averaged with exponential decay weighting.

print("Temporal Ensemble:")
print("  Chunk 1: t=0..99")
print("  Chunk 2:        t=50..149")
print("  Overlap: t=50..99 averaged with exp(-Δt/τ) weights")


### Key Takeaway

ACT represents actions as **continuous vectors with learned distributions**. No tokenization, no discretization. The CVAE handles action multimodality. This was the dominant paradigm before VLAs entered the picture.
